# step02_suna_colocation - SUNA Stage 2: co-location with the CTD

Stages 0 and 1 worked on SUNA time series. Stage 2 gives every SUNA nitrate frame a **pressure** by matching
it in time to the CTD, and tags it with the **cast** it belongs to. This is what turns a continuous nitrate log
into nitrate-vs-pressure profiles, and it is the step that splits the continuous P45_06 logs into casts without
any guessing, because the CTD record is the authority.

**Why time-matching is the right method, and why it works here.** The SUNA and the CTD are on the same rosette
and sample on their own clocks. If those clocks agree, then at any instant both instruments are at the same
depth, so the CTD pressure at a SUNA frame's timestamp IS that frame's pressure. We checked the clocks on cast
06: the CTD pressure maximum (168 db at 14:44:09) and the SUNA nitrate maximum (14:44:17) agree to 8 seconds,
within one SUNA sample. So no clock correction is needed; a configurable `CLOCK_OFFSET_S` is provided in case a
later cast shows drift, and the notebook reports the offset per cast as a QC diagnostic.

**How a cast is defined.** Each CTD `.cnv` is one cast, with a `start_time` header and a `timeS` elapsed-seconds
channel, so every scan has an absolute UTC time. A SUNA frame belongs to a cast if its time falls inside that
cast's window (optionally padded). Frames outside every cast window are the between-cast surface baseline from
Stage 0; they are kept in the output and marked `in_cast = False`, not deleted. Within a cast, the pressure
maximum splits downcast from upcast.

**What each matched frame carries:** the SUNA nitrate and the Stage 1 `L1_clean` verdict, plus the CTD pressure,
temperature and conductivity interpolated to the SUNA time. Temperature and conductivity are carried now because
the next stage (re-derivation) needs in-situ temperature and salinity for the Sakamoto temperature-compensated,
salinity-subtracted correction (Sakamoto, Johnson and Coletti, 2009; Sakamoto et al., 2017).

**What Stage 2 does NOT do:** it does not re-derive nitrate from the spectra and does not apply the bottle-bias
offset (your profiles still sit on the raw SUNA offset, near -3 uM at the surface). Those are later stages. Here
the job is purely geometric: put each nitrate value at the right pressure, in the right cast.

Sources: Johnson et al. (2018), *JTECH* (BGC-Argo nitrate processing, time/pressure alignment); Sakamoto,
Johnson and Coletti (2009) and Sakamoto et al. (2017) (temperature and pressure corrections, for the next stage);
Sea-Bird SUNA V2 and SBE processing documentation.

In [ ]:
import re, sys
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np, pandas as pd
import matplotlib

INTERACTIVE_ZOOM = True
try:
    get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_ZOOM else "inline")
except Exception:
    INTERACTIVE_ZOOM = False
    try: get_ipython().run_line_magic("matplotlib", "inline")
    except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
if INTERACTIVE_ZOOM:
    try: plt.ioff()
    except Exception: pass
try:
    from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image = None

# ------------------------------------------------------------------ resolve repo root
try:
    _anchor = Path(__file__).resolve()
except NameError:
    _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists():
        sys.path.insert(0, str(_p)); break
from project_paths import project_root, find_suna_cal

CRUISE_ID  = "P45_06"
CTD_ROOT   = project_root()
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
SUNA_L0    = CRUISE_DIR / "L0" / "SUNA"
CTD_CNV    = CRUISE_DIR / "L1" / "CTD"          # processed per-cast .cnv files
CAL_FILE   = find_suna_cal(cruise_id=CRUISE_ID, filename="SNA2503C.cal")
OUT_DIR    = CRUISE_DIR / "_audit" / "suna"; OUT_DIR.mkdir(parents=True, exist_ok=True)
L2_DIR     = CRUISE_DIR / "L2" / "SUNA";     L2_DIR.mkdir(parents=True, exist_ok=True)

NSPEC = 256; SPEC_START = 11; FIT_LO, FIT_HI = 217.0, 240.0
BAD_FLAG = -9.99e-29
SAT_COUNTS=60000; LOW_COUNTS=2000; RMSE_HI=1.0e-3; HUM_HI=10.0; WARMUP_S=60.0
PEAK_LO,PEAK_HI=20000,58000; ABS254_LO,ABS254_HI=-0.60,-0.15; SPECAVG_LO,SPECAVG_HI=8000,30000
LONGWAVE_LO,LONGWAVE_HI=310.0,360.0; LONGWAVE_A_HI=0.10; SHAPE_LO,SHAPE_HI=250.0,360.0; SHAPE_DEV_HI=0.01
CLOCK_OFFSET_S = 0.0
CAST_PAD_S     = 0.0
MAX_MATCH_GAP_S= 5.0
EXPLORER_FIGSIZE=(12.5,6.0)
print("cruise:", CRUISE_ID, "| SUNA:", SUNA_L0, "| CTD cnv:", CTD_CNV, "| L2 out:", L2_DIR)

In [ ]:
# ------------------------------------------------------------------ CTD .cnv reader
def parse_cnv(path):
    """Return dict(cast, start_time, df with time_utc + mapped channels)."""
    lines = Path(path).read_text(errors="replace").splitlines()
    names, start, iend = [], None, None
    for i, ln in enumerate(lines):
        if ln.startswith("# name"):
            names.append(ln.split("=", 1)[1].split(":")[0].strip())
        elif ln.startswith("# start_time"):
            s = ln.split("=", 1)[1].split("[")[0].strip()
            for fmt in ("%b %d %Y %H:%M:%S", "%b %d %Y  %H:%M:%S"):
                try: start = datetime.strptime(s, fmt); break
                except ValueError: pass
        elif ln.strip() == "*END*":
            iend = i + 1; break
    data = []
    for ln in lines[iend:]:
        p = ln.split()
        if len(p) >= len(names):
            try: data.append([float(x) for x in p[:len(names)]])
            except ValueError: pass
    arr = np.array(data, float)
    arr[(arr <= -8e-29) & (arr >= -1.2e-28)] = np.nan   # bad-flag mask
    df = pd.DataFrame(arr, columns=names)
    # map channels by name
    def pick(subs):
        for s in subs:
            for c in names:
                if c.lower().startswith(s) or s in c.lower(): return c
        return None
    tcol = pick(["times", "time, elapsed"])
    pcol = pick(["prdm", "prdm", "pressure"])
    Tcol = pick(["tv290c", "t090c", "tv290", "temperature"])
    Ccol = pick(["c0s/m", "cond"])
    if start is None or tcol is None or pcol is None:
        raise ValueError(f"{Path(path).name}: missing start_time/time/pressure (start={start}, t={tcol}, p={pcol})")
    out = pd.DataFrame({"time_utc": [start + timedelta(seconds=s) for s in df[tcol].to_numpy()],
                        "pressure_db": df[pcol].to_numpy()})
    out["temp_C"] = df[Tcol].to_numpy() if Tcol else np.nan
    out["cond_Sm"] = df[Ccol].to_numpy() if Ccol else np.nan
    return dict(cast=Path(path).stem, start=start, df=out,
                chans=dict(time=tcol, pressure=pcol, temp=Tcol, cond=Ccol))

def load_ctd_casts(cnv_dir):
    casts = {}
    for p in sorted(cnv_dir.glob("*.cnv")):
        try: casts[p.stem] = parse_cnv(p)
        except Exception as e: print("  skip", p.name, "-", e)
    return casts

In [ ]:
# ------------------------------------------------------------------ SUNA reader + QC (same as step01)
def parse_cal(path):
    wl=[]
    for ln in Path(path).read_text(errors="replace").splitlines():
        p=ln.split(",")
        if p[0]=="E" and len(p)>=2:
            try: wl.append(float(p[1]))
            except ValueError: pass
    return np.array(wl,float)

def _ts(datef,timed):
    y=int(float(datef))//1000; doy=int(float(datef))%1000
    return datetime(y,1,1)+timedelta(days=doy-1,hours=float(timed))

def parse_suna_file(path):
    fhr,rows,spec={},[],[]
    for ln in Path(path).read_text(errors="replace").splitlines():
        if ln.startswith("SATFHR"):
            p=ln.split(",",2)
            if len(p)==3: fhr[p[1].strip()]=p[2].strip()
        elif ln.startswith("SATSLF"):
            f=ln.split(",")
            if len(f)<SPEC_START+NSPEC+6: continue
            try: sp=np.array(f[SPEC_START:SPEC_START+NSPEC],float)
            except ValueError: continue
            tr=f[SPEC_START+NSPEC:]
            def gg(i):
                try: return float(tr[i])
                except (ValueError,IndexError): return np.nan
            try:
                rows.append(dict(time_utc=_ts(f[1],f[2]),nitrate_uM=float(f[3]),abs254=float(f[5]),
                    abs350=float(f[6]),spec_avg=float(f[8]),dark=float(f[9]),
                    humidity=gg(4),fit_rmse=gg(13)))
                spec.append(sp)
            except ValueError: continue
    return fhr,pd.DataFrame(rows),(np.array(spec) if spec else np.empty((0,NSPEC)))

def qc_frames(df,spec,wl):
    if df.empty: return df
    df=df.copy(); secs=(df["time_utc"]-df["time_utc"].iloc[0]).dt.total_seconds().to_numpy()
    peak=spec.max(axis=1); df["peak_counts"]=peak
    df["f_warmup"]=secs<WARMUP_S; df["f_saturated"]=peak>SAT_COUNTS; df["f_lowsignal"]=peak<LOW_COUNTS
    df["f_highrmse"]=df["fit_rmse"]>RMSE_HI; df["f_humid"]=df["humidity"]>HUM_HI
    l0=df[["f_warmup","f_saturated","f_lowsignal","f_highrmse","f_humid"]].any(axis=1).to_numpy()
    ref=np.median(spec[~l0],axis=0) if (~l0).sum()>=5 else np.median(spec,axis=0)
    dk=float(np.median(df["dark"])); A=-np.log10(np.clip(spec-dk,1,None)/np.clip(ref-dk,1,None))
    lw=(wl>=LONGWAVE_LO)&(wl<=LONGWAVE_HI); sb=(wl>=SHAPE_LO)&(wl<=SHAPE_HI); fb=(wl>=FIT_LO)&(wl<=FIT_HI)
    df["longwave_A"]=np.mean(np.abs(A[:,lw]),axis=1)
    rb=ref[sb]-ref[sb].mean(); sbn=spec[:,sb]-spec[:,sb].mean(axis=1,keepdims=True)
    df["shape_dev"]=1.0-(sbn@rb)/(np.sqrt((sbn**2).sum(1))*np.sqrt((rb**2).sum())+1e-9)
    df["f_outofwater"]=~(df["peak_counts"].between(PEAK_LO,PEAK_HI)&df["abs254"].between(ABS254_LO,ABS254_HI)
                         &df["spec_avg"].between(SPECAVG_LO,SPECAVG_HI))
    df["f_satfit"]=spec[:,fb].max(axis=1)>=SAT_COUNTS
    df["f_baseline"]=df["longwave_A"]>LONGWAVE_A_HI; df["f_shape"]=df["shape_dev"]>SHAPE_DEV_HI
    FATAL=["f_warmup","f_saturated","f_lowsignal","f_highrmse","f_outofwater","f_satfit","f_baseline","f_shape"]
    df["L1_clean"]=~df[FATAL].any(axis=1)
    return df

def load_all_suna(wl):
    frames=[]
    for p in sorted(set(list(SUNA_L0.glob("*.CSV"))+list(SUNA_L0.glob("*.csv")))):
        fhr,df,sp=parse_suna_file(p)
        if df.empty: continue
        df=qc_frames(df,sp,wl); df["suna_file"]=p.name
        frames.append(df[["time_utc","nitrate_uM","L1_clean","suna_file","longwave_A","shape_dev"]])
    return pd.concat(frames,ignore_index=True).sort_values("time_utc").reset_index(drop=True) if frames else pd.DataFrame()

In [ ]:
# ------------------------------------------------------------------ co-locate SUNA -> CTD
CAL_WL = parse_cal(CAL_FILE)
if CAL_WL.size!=NSPEC: CAL_WL=np.arange(NSPEC,dtype=float)
CASTS = load_ctd_casts(CTD_CNV)
SUNA  = load_all_suna(CAL_WL)
print(f"{len(CASTS)} CTD casts | {len(SUNA)} SUNA frames\n")

def _epoch(s): return (pd.to_datetime(s).astype("int64")/1e9).to_numpy()

def colocate():
    if SUNA.empty or not CASTS: 
        print("missing SUNA or CTD data"); return pd.DataFrame(), pd.DataFrame()
    st = SUNA.copy()
    st["time_match"] = st["time_utc"] + pd.to_timedelta(CLOCK_OFFSET_S, unit="s")
    st_ep = _epoch(st["time_match"])
    st["pressure_db"]=np.nan; st["temp_C"]=np.nan; st["cond_Sm"]=np.nan
    st["cast"]=""; st["in_cast"]=False; st["leg"]=""
    diag=[]
    for cid,c in CASTS.items():
        cdf=c["df"]; cep=_epoch(cdf["time_utc"]); p=cdf["pressure_db"].to_numpy()
        t0,t1=cep[0]-CAST_PAD_S, cep[-1]+CAST_PAD_S
        sel=(st_ep>=t0)&(st_ep<=t1)
        if sel.sum()==0: continue
        idx=np.searchsorted(cep,st_ep[sel]); idx=np.clip(idx,1,len(cep)-1)
        lo=cep[idx-1]; hi=cep[idx]; w=(st_ep[sel]-lo)/np.where(hi-lo==0,1,hi-lo)
        def interp(col):
            v=cdf[col].to_numpy(); return v[idx-1]*(1-w)+v[idx]*w
        near=np.minimum(np.abs(st_ep[sel]-lo),np.abs(st_ep[sel]-hi))
        pr=interp("pressure_db"); pr[near>MAX_MATCH_GAP_S]=np.nan
        ii=np.where(sel)[0]
        st.loc[st.index[ii],"pressure_db"]=pr
        st.loc[st.index[ii],"temp_C"]=interp("temp_C")
        st.loc[st.index[ii],"cond_Sm"]=interp("cond_Sm")
        st.loc[st.index[ii],"cast"]=cid; st.loc[st.index[ii],"in_cast"]=True
        # down/up split at CTD pressure max time
        tmax=cep[np.nanargmax(p)]
        st.loc[st.index[ii],"leg"]=np.where(st_ep[sel]<=tmax,"down","up")
        # clock-offset diagnostic: SUNA nitrate-max vs CTD pressure-max
        sw=st.loc[st.index[ii]]; swc=sw[sw["L1_clean"]]
        if len(swc)>5:
            nmax_t=_epoch(pd.Series([swc.loc[swc["nitrate_uM"].idxmax(),"time_match"]]))[0]
            diag.append(dict(cast=cid, n_suna=int(sel.sum()), n_clean=int(sw["L1_clean"].sum()),
                p_max_db=round(float(np.nanmax(p)),1),
                suna_minus_ctd_s=round(nmax_t-tmax,1)))
    return st, pd.DataFrame(diag)

L2, DIAG = colocate()
if not L2.empty:
    L2.to_csv(L2_DIR/f"{CRUISE_ID}__suna_L2_colocated.csv", index=False, encoding="utf-8-sig")
    nin=int(L2["in_cast"].sum()); ncl=int((L2["in_cast"]&L2["L1_clean"]).sum())
    print(f"matched {nin}/{len(L2)} frames to casts | {ncl} are in-cast AND L1-clean")
    print("between-cast frames kept (in_cast=False):", int((~L2["in_cast"]).sum()))
    if not DIAG.empty:
        print("\nper-cast co-location (clock-offset diagnostic = SUNA nitrate-max minus CTD pressure-max):")
        with pd.option_context("display.width",200): print(DIAG.to_string(index=False))
        print(f"\nmedian clock offset across casts: {DIAG['suna_minus_ctd_s'].median():.1f} s "
              f"(apply via CLOCK_OFFSET_S only if large/consistent)")
    print("\nL2 ->", L2_DIR)

In [ ]:
# ------------------------------------------------------------------ per-cast nitrate-vs-pressure profiles
def profile_png(cid):
    d = L2[(L2["cast"]==cid) & L2["L1_clean"] & L2["pressure_db"].notna()]
    if d.empty: return None
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig=Figure(figsize=(5.2,6.4)); FigureCanvasAgg(fig); ax=fig.subplots()
    for leg,c,m in [("down","#2E6FB0","."),("up","#E07B39","x")]:
        g=d[d["leg"]==leg]
        if len(g): ax.plot(g["nitrate_uM"],g["pressure_db"],m,ms=3.5,color=c,label=leg,alpha=0.8)
    ax.invert_yaxis(); ax.set_xlabel("NO3 (uM, raw SUNA offset)"); ax.set_ylabel("pressure (db)")
    ax.axvline(0,color="#bbb",lw=0.6); ax.legend(fontsize=8,frameon=False)
    ax.set_title(f"{CRUISE_ID}  {cid}  ({len(d)} clean frames)",fontsize=10)
    for s in ("top","right"): ax.spines[s].set_visible(False)
    fig.tight_layout()
    out=OUT_DIR/f"{CRUISE_ID}__suna_profile__{cid}.png"; fig.savefig(out,dpi=130)
    try: plt.close(fig)
    except Exception: pass
    return out

if not L2.empty:
    for cid in [c for c in CASTS if ((L2["cast"]==c)&L2["L1_clean"]).any()]:
        png=profile_png(cid)
        if png and Image is not None:
            try: display(Image(filename=str(png)))
            except Exception: pass
        if png: print("  saved", png.name)

In [ ]:
# ------------------------------------------------------------------ interactive: cast profile + time context
try:
    import ipywidgets as W
    _HAVE_W=True
except Exception:
    _HAVE_W=False; print("ipywidgets not installed; pip install ipywidgets ipympl then Restart + Run All.")

def _cast_png_bytes(cid):
    import io
    d=L2[L2["cast"]==cid]; dc=d[d["L1_clean"]&d["pressure_db"].notna()]
    c=CASTS[cid]["df"]; ct=(c["time_utc"]-c["time_utc"].iloc[0]).dt.total_seconds()
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig=Figure(figsize=EXPLORER_FIGSIZE); FigureCanvasAgg(fig)
    ax1=fig.add_subplot(1,2,1); ax2=fig.add_subplot(2,2,2); ax3=fig.add_subplot(2,2,4,sharex=ax2)
    for leg,col,m in [("down","#2E6FB0","."),("up","#E07B39","x")]:
        g=dc[dc["leg"]==leg]
        if len(g): ax1.plot(g["nitrate_uM"],g["pressure_db"],m,ms=3.5,color=col,label=leg,alpha=0.8)
    ax1.invert_yaxis(); ax1.axvline(0,color="#bbb",lw=0.6); ax1.set_xlabel("NO3 (uM, raw)")
    ax1.set_ylabel("pressure (db)"); ax1.set_title(f"{cid} profile ({len(dc)} clean)",fontsize=10)
    ax1.legend(fontsize=8,frameon=False)
    dt0=d["time_utc"].iloc[0] if len(d) else c["time_utc"].iloc[0]
    st=(d["time_utc"]-dt0).dt.total_seconds()
    ax2.plot((c["time_utc"]-dt0).dt.total_seconds(),c["pressure_db"],lw=0.8,color="#444")
    ax2.invert_yaxis(); ax2.set_ylabel("CTD p (db)"); ax2.set_title("time context",fontsize=9)
    gc=d[d["L1_clean"]]; gf=d[~d["L1_clean"]]
    ax3.plot(st,d["nitrate_uM"],lw=0.4,color="#ccc")
    ax3.plot((gc["time_utc"]-dt0).dt.total_seconds(),gc["nitrate_uM"],".",ms=2,color="#2E6FB0",label="clean")
    ax3.plot((gf["time_utc"]-dt0).dt.total_seconds(),gf["nitrate_uM"],".",ms=2.5,color="#d62728",label="flagged")
    ax3.axhline(0,color="#bbb",lw=0.5); ax3.set_xlabel("s from cast start"); ax3.set_ylabel("NO3 (uM)")
    ax3.legend(fontsize=7,frameon=False,ncol=2)
    for a in (ax1,ax2,ax3):
        for s in ("top","right"): a.spines[s].set_visible(False)
    fig.suptitle(f"{CRUISE_ID}  {cid}",fontsize=11); fig.tight_layout(rect=[0,0,1,0.96])
    buf=io.BytesIO(); fig.savefig(buf,format="png",dpi=118); buf.seek(0)
    try: plt.close(fig)
    except Exception: pass
    return buf.getvalue()

if _HAVE_W and not L2.empty:
    cids=[c for c in CASTS if (L2["cast"]==c).any()]
    wc=W.Dropdown(options=cids,value=cids[0],description="cast")
    def _r(cid): display(Image(data=_cast_png_bytes(cid)))
    out=W.interactive_output(_r,{"cid":wc}); display(W.VBox([wc,out]))